# LinkedIn post classifier

Train a binary classifier: **useful = 1**, **useless = 0**.

In [21]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences


training_useful = "../data/useful_posts.csv"
training_useless = "../data/useless_posts.csv"

VectorSize = 300


## 1. Load and clean

Each post is one example, regardless of word count. Lowercasing and removing punctuation make the vocabulary simpler. The trade-off is that names like `C++` lose their symbols.


In [22]:
useful_train = pd.read_csv(training_useful)['post_content'].dropna().astype(str).tolist()
useless_train = pd.read_csv(training_useless)['post_content'].dropna().astype(str).tolist()

def clean(text):
    return ' '.join(re.findall(r'\b\w+\b', text.lower()))

texts = [clean(text) for text in useful_train + useless_train]
labels = np.array([1] * (len(useful_train)) + [0] * (len(useless_train)))
print(f'{len(useful_train)} useful, {len(useless_train)} useless posts')


618 useful, 568 useless posts


## 2. Split and encode

Reserve 20% for test, then 20% of the remaining data for validation (16% overall). `stratify` keeps class proportions similar in all three sets. Fit the tokenizer on training text only so validation and test vocabulary do not leak into training. Padding gives every post the same input length; longer posts are cut off.


In [23]:
train_val_texts, test_texts, y_train_val, y_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)
train_texts, val_texts, y_train, y_val = train_test_split(
    train_val_texts, y_train_val, test_size=0.2, stratify=y_train_val, random_state=42
)
MAX_LEN = 350  # Room for most approximately 300-word posts.
tokenizer = Tokenizer(oov_token='<OOV>', filters='')
tokenizer.fit_on_texts(train_texts)

def encode(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

x_train, x_val, x_test = encode(train_texts), encode(val_texts), encode(test_texts)


## 3. Train Word2Vec

Word2Vec learns a 100-number vector for each word from its neighbors in the training posts. `min_count=1` keeps rare technical words, but also typos. The embedding matrix aligns vectors with tokenizer IDs; row 0 is padding.


In [24]:
w2v = Word2Vec([text.split() for text in train_texts], vector_size=VectorSize, min_count=0, epochs=10)
embeddings = np.zeros((len(tokenizer.word_index) + 1, 300), dtype=np.float32)
for word, index in tokenizer.word_index.items():
    if word in w2v.wv:
        embeddings[index] = w2v.wv[word]


## 4. Train the CNN

`Conv1D` looks for patterns across five neighboring words. Pooling retains strong pattern signals. The sigmoid output is the estimated probability that a post is useful. Training the embeddings lets them adapt to this task, but may overfit; early stopping restores the weights from the best validation-loss epoch.


In [25]:
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(len(embeddings), VectorSize, weights=[embeddings], trainable=True),
    Conv1D(64, 20, activation='relu'),
    MaxPooling1D(2),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(
    x_train, y_train, validation_data=(x_val, y_val),
    epochs=20, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
)


Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 277ms/step - accuracy: 0.5277 - loss: 0.7387 - val_accuracy: 0.5368 - val_loss: 0.6594
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 131ms/step - accuracy: 0.6557 - loss: 0.6385 - val_accuracy: 0.6263 - val_loss: 0.6500
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 131ms/step - accuracy: 0.6794 - loss: 0.5975 - val_accuracy: 0.5474 - val_loss: 0.6857
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 131ms/step - accuracy: 0.7203 - loss: 0.5579 - val_accuracy: 0.6000 - val_loss: 0.6422
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 136ms/step - accuracy: 0.7863 - loss: 0.4835 - val_accuracy: 0.6053 - val_loss: 0.6601
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 135ms/step - accuracy: 0.8786 - loss: 0.3900 - val_accuracy: 0.6000 - val_loss: 0.6577


## 5. Evaluate and use

Accuracy is the overall fraction correct. Precision and recall show how the model performs for each class. A probability of 0.5 or higher is classified as useful.


In [26]:
# x_test and y_test were held out from the combined dataset in the split cell.
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
probabilities = model(x_test, training=False).numpy().ravel()
predictions = (probabilities >= 0.5).astype(int)

print(f"Test accuracy: {accuracy:.1%}")
print(classification_report(y_test, predictions, target_names=["useless", "useful"]))


Test accuracy: 61.3%
              precision    recall  f1-score   support

     useless       0.64      0.43      0.52       114
      useful       0.60      0.78      0.68       124

    accuracy                           0.61       238
   macro avg       0.62      0.61      0.60       238
weighted avg       0.62      0.61      0.60       238



In [28]:
from pathlib import Path

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

Path( "../exports/linkedin_posts_classifier.tflite").write_bytes(converter.convert())

INFO:tensorflow:Assets written to: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpubefrv47/assets


INFO:tensorflow:Assets written to: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpubefrv47/assets


Saved artifact at '/var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpubefrv47'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 350), dtype=tf.float32, name='keras_tensor_14')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  6242914320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6294941776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6294935632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6294941008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6294936592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6294940816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6294942544: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1793250753.049815  841465 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1793250753.049837  841465 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1793250753.049975  841465 reader.cc:83] Reading SavedModel from: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpubefrv47
I0000 00:00:1793250753.050422  841465 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1793250753.050426  841465 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpubefrv47
I0000 00:00:1793250753.054693  841465 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1793250753.082335  841465 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpubefrv47
I0000 00:00:1793250753.090905  841465 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 40933 microseconds.


8985368

In [29]:
from io import BytesIO
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from google.cloud import storage

project_id = "linkedin-training-512914"
bucket_name = "linkedin-training-512914-models"

# Shared default for users who have not trained a personal model.
bundle = BytesIO()
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    archive.writestr("model.tflite", Path("../exports/linkedin_posts_classifier.tflite").read_bytes())
    archive.writestr("tokenizer.json", tokenizer.to_json())

bucket = storage.Client(project=project_id).bucket(bucket_name)
bucket.blob("models/global/v1/model.bundle.zip").upload_from_string(
    bundle.getvalue(),
    content_type="application/zip",
)

In [30]:
from pathlib import Path

Path("../exports/tokenizer.json").write_text(tokenizer.to_json())

555238